#  Feature Engineering — NPS Satisfaction Score Prediction


## Contexte

Le dataset IBM Telco contient 54 colonnes brutes.  
L'objectif du feature engineering est d'enrichir ce dataset avec des variables  
qui capturent mieux le comportement et le profil des clients pour prédire  
leur **Satisfaction Score** (variable cible ordinale — 3 classes).

---

## Architecture — Strategy Pattern

Chaque groupe de features est encapsulé dans une **stratégie** indépendante.  
Le contexte `FeatureEngineer` applique toutes les stratégies dans l'ordre.

```
FeatureEngineer(strategies=[...])
        │
        ├── ValueFeatures()        → valeur financière
        ├── EngagementFeatures()   → services & usage
        ├── ContractFeatures()     → contrat & engagement
        ├── TenureFeatures()       → ancienneté
        └── DemographicFeatures()  → profil démographique
```

> `EngagementFeatures` doit être appliqué **avant** `ContractFeatures`  
> car `Locked_Value` dépend de `Services_Count`.

---

## Utilisation

```python
from src.feature_engineering import (
    FeatureEngineer,
    ValueFeatures,
    EngagementFeatures,
    ContractFeatures,
    TenureFeatures,
    DemographicFeatures
)

engineer = FeatureEngineer(strategies=[
    ValueFeatures(),
    EngagementFeatures(),
    ContractFeatures(),
    TenureFeatures(),
    DemographicFeatures()
])

df_enriched = engineer.run(df)

# Sauvegarder
df_enriched.to_csv("data/processed/telco_featured.csv", index=False)
```

---

## 1. ValueFeatures — Valeur Financière

### `Revenue_Per_Month`

$$\text{Revenue Per Month} = \frac{\text{Total Revenue}}{\text{Tenure in Months}}$$

Revenu moyen mensuel réel du client sur toute sa durée de vie.  
Permet de comparer des clients d'anciennetés différentes sur la même échelle.

---

### `Charge_Evolution`

$$\text{Charge Evolution} = \text{Monthly Charge} - \text{Revenue Per Month}$$

Écart entre le tarif actuel et la moyenne historique.

| Valeur | Signification |
|---|---|
| **Positive** | Le client a upgradé son forfait |
| **Négative** | Le client a eu une réduction ou downgrade |
| **≈ 0** | Forfait stable depuis le début |

---

### `Refund_Rate`

$$\text{Refund Rate} = \frac{\text{Total Refunds}}{\text{Total Revenue}}$$

Taux de remboursement — proxy d'insatisfaction.  
Un client remboursé fréquemment a probablement eu des problèmes de service.

---

### `Has_Been_Refunded`

```
1 → le client a déjà été remboursé
0 → aucun remboursement
```

Flag binaire complémentaire au `Refund_Rate`.

---

### `Billing_Complexity`

```
Billing_Complexity = (Extra Data Charges > 0) + (Long Distance Charges > 0) + (Refunds > 0)
```

Score de 0 à 3 — plus la facture est complexe, plus le client risque d'être confus ou insatisfait.

---

## 2. EngagementFeatures — Services & Usage

### `Services_Count`

Nombre total de services souscrits parmi :  
`Online Security`, `Online Backup`, `Device Protection Plan`, `Premium Tech Support`,  
`Streaming TV`, `Streaming Movies`, `Streaming Music`, `Unlimited Data`, `Multiple Lines`

Plus ce score est élevé, plus le client est engagé et dépendant des services.

---

### `Streaming_Count`

Nombre de services streaming souscrits parmi :  
`Streaming TV`, `Streaming Movies`, `Streaming Music` (0 à 3)

---

### `Security_Score`

Nombre de services de sécurité/protection souscrits parmi :  
`Online Security`, `Online Backup`, `Device Protection Plan`, `Premium Tech Support` (0 à 4)

---

### `Full_Protection_Score`

Score global de protection — identique au `Security_Score`.  
Un score élevé indique un client qui investit dans la sécurité → attentes élevées.

---

### `Support_Profile`

```
1 → Premium Tech Support = Yes ET Device Protection Plan = Yes
0 → sinon
```

Client avec double protection — a souscrit au support prioritaire ET à la protection équipement.

---

### `Service_Diversity_Score`

```
Service_Diversity_Score = Streaming_Count + Full_Protection_Score 
                        + (Multiple Lines == Yes) + (Unlimited Data == Yes)
```

Score global d'engagement multi-services.

| Score | Profil |
|---|---|
| 0-2 | Client basique — peu de services |
| 3-5 | Client intermédiaire |
| 6+ | Client premium — attentes très élevées |

---

### `Referral_Rate`

$$\text{Referral Rate} = \frac{\text{Number of Referrals}}{\text{Tenure in Months}}$$

Taux de recommandation par mois — capture l'ambassadorat actif du client.

---

### `Heavy_Internet_User`

```
1 → Avg Monthly GB Download > 75ème percentile
0 → sinon
```

Gros consommateur de data — profil streaming/gaming.

---

### `Internet_Value`

$$\text{Internet Value} = \text{Internet Type Score} \times \text{Avg Monthly GB Download}$$

Avec : DSL = 1, Cable = 2, Fiber Optic = 3

Combine la qualité de connexion et l'intensité d'usage.

---

## 3. ContractFeatures — Contrat & Engagement

### `Contract_Encoded`

Encodage ordinal du type de contrat :

| Contrat | Valeur |
|---|---|
| Month-to-Month | 0 |
| One Year | 1 |
| Two Year | 2 |

---

### `Engagement_Score`

$$\text{Engagement Score} = \text{Contract Encoded} \times \text{Tenure in Months}$$

Un client Two Year depuis 60 mois est bien plus engagé qu'un Two Year depuis 3 mois.

---

### `Contract_Risk`

```
1 → Contract = Month-to-Month ET Tenure < 12 mois
0 → sinon
```

Nouveau client sans engagement → risque maximum de churn.

---

### `Contract_Tenure_Mismatch`

```
1 → Contract = Month-to-Month ET Tenure > 24 mois
0 → sinon
```

Client fidèle depuis plus de 2 ans toujours sans engagement →  
il n'a jamais voulu s'engager malgré sa loyauté → profil intéressant à analyser.

---

### `Locked_Value`

$$\text{Locked Value} = \text{Contract Encoded} \times \text{Services Count}$$

>  Nécessite `Services_Count` — appliquer `EngagementFeatures` avant `ContractFeatures`.

Un client Two Year avec beaucoup de services est très difficile à perdre.

---

## 4. TenureFeatures — Ancienneté

### `Tenure_Segment`

Segmentation de l'ancienneté en 4 groupes ordonnés :

| Valeur | Segment | Tenure | Profil |
|---|---|---|---|
| 0 | Nouveau | 0-12 mois | Période critique — taux de churn maximal |
| 1 | Développement | 12-24 mois | Client qui commence à s'installer |
| 2 | Etabli | 24-48 mois | Client stable, habitudes formées |
| 3 | Fidèle | 48-72 mois | Ambassadeur potentiel |

---

## 5. DemographicFeatures — Profil Démographique

### `Vulnerable_Senior`

```
1 → Senior Citizen = Yes ET Premium Tech Support = No
0 → sinon
```

Senior sans support premium → risque de difficultés techniques → source d'insatisfaction.

---

### `Family_Profile`

```
1 → Married = Yes ET Dependents = Yes
0 → sinon
```

Client en famille — besoins spécifiques : data illimitée, multi-lignes, streaming.

---

### `Digital_Profile`

```
1 → Paperless Billing = Yes ET Payment Method = Bank Withdrawal ET Internet Service = Yes
0 → sinon
```

Client full digital — profil technophile avec attentes élevées sur la qualité du service.

---

### `High_Value_At_Risk`

```
1 → CLTV > 75ème percentile ET Tenure < 12 mois ET Contract = Month-to-Month
0 → sinon
```

Client à haute valeur financière, récent et sans engagement → signal critique de rétention.

---

## Synthèse Complète

| # | Feature | Stratégie | Justification |
|---|---|---|---|
| 1 | `Revenue_Per_Month` | ValueFeatures | Valeur mensuelle réelle normalisée |
| 2 | `Charge_Evolution` | ValueFeatures | Évolution du comportement d'achat |
| 3 | `Refund_Rate` | ValueFeatures | Proxy d'insatisfaction |
| 4 | `Has_Been_Refunded` | ValueFeatures | Flag de problème de service |
| 5 | `Billing_Complexity` | ValueFeatures | Complexité de facturation → confusion |
| 6 | `Services_Count` | EngagementFeatures | Engagement global |
| 7 | `Streaming_Count` | EngagementFeatures | Usage streaming |
| 8 | `Security_Score` | EngagementFeatures | Investissement sécurité |
| 9 | `Full_Protection_Score` | EngagementFeatures | Score protection global |
| 10 | `Support_Profile` | EngagementFeatures | Double protection souscrite |
| 11 | `Service_Diversity_Score` | EngagementFeatures | Score multi-services |
| 12 | `Referral_Rate` | EngagementFeatures | Ambassadorat actif |
| 13 | `Heavy_Internet_User` | EngagementFeatures | Gros consommateur data |
| 14 | `Internet_Value` | EngagementFeatures | Qualité × usage internet |
| 15 | `Contract_Encoded` | ContractFeatures | Encodage ordinal contrat |
| 16 | `Engagement_Score` | ContractFeatures | Contrat × ancienneté |
| 17 | `Contract_Risk` | ContractFeatures | Nouveau + sans engagement |
| 18 | `Contract_Tenure_Mismatch` | ContractFeatures | Fidèle sans engagement |
| 19 | `Locked_Value` | ContractFeatures | Contrat × services |
| 20 | `Tenure_Segment` | TenureFeatures | Segmentation ancienneté |
| 21 | `Vulnerable_Senior` | DemographicFeatures | Senior sans support |
| 22 | `Family_Profile` | DemographicFeatures | Profil famille |
| 23 | `Digital_Profile` | DemographicFeatures | Profil full digital |
| 24 | `High_Value_At_Risk` | DemographicFeatures | Client précieux à risque |

---

## Features Exclues

| Feature | Raison |
|---|---|
| `Churn Score` | Donnée future — data leakage |
| `Churn Score Category` | Doublon du Churn Score |
| `CLTV Category` | Doublon du CLTV numérique |
| `Churn Label` | Doublon de Churn Value |
| `Total Charges / Tenure` | Redondant avec `Monthly Charge` |

---

## Sources

- Analyse exploratoire du dataset IBM Telco (7043 clients, Californie Q3)
- Littérature : feature engineering télécom — patterns d'usage, engagement contractuel,  
  segmentation démographique (cf. références EDA)